# Step 5b — Scoring & Portfolio Impact

**What this notebook does:** takes Step 5a's output (the blind estimator's demand_characteristics.csv, unaltered) and (1) scores it against ground truth -- available only in this Project's own environment, never given to the blind chat -- and (2) reports where any low-confidence SKUs concentrate in terms of plant, line, category, volume and revenue.

**This notebook does no estimation.** It only reports on results Step 5a already produced. Logic lives in `src/portfolio_impact.py`.

**Prerequisite:** `demand_characteristics.csv` from a Step 5a run, and this session's `data_primary/_truth/ground_truth.csv` (from step02, run earlier in this same Project's Colab).

## Setup — clone the repo

In [ ]:
import subprocess, os, sys

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

REPO = '/content/ibp-tradeoff'
os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO)
sys.path.insert(0, REPO)
print('Working directory:', os.getcwd())


## Get the raw + clean data
If step02 and step04 already ran in this session, this reuses that. Otherwise regenerates from scratch (same approved pipeline).

In [ ]:
import pandas as pd
from src.ingest import DataIngestor
from src.cleaner import DataCleaner

if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py')
    sh('mv data data_primary')

clean_path = 'data_primary/clean/clean_master.parquet'
if os.path.exists(clean_path):
    clean_master = pd.read_parquet(clean_path)
    ing = DataIngestor(repo_root=REPO, data_root='data_primary')
    raw = ing.load()
    cl = DataCleaner(ing.schema, ing.assumptions)
    _, sku_master, _ = cl.clean(raw)
else:
    ing = DataIngestor(repo_root=REPO, data_root='data_primary')
    raw = ing.load()
    cl = DataCleaner(ing.schema, ing.assumptions)
    clean_master, sku_master, _ = cl.clean(raw)
    os.makedirs('data_primary/clean', exist_ok=True)
    clean_master.to_parquet(clean_path, index=False)

print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)


## Upload Step 5a's output
Select `demand_characteristics.csv` (from the blind chat's notebook run).

In [ ]:
from google.colab import files
print('Select demand_characteristics.csv:')
uploaded = files.upload()
dc_name = list(uploaded.keys())[0]
demand_characteristics = pd.read_csv(dc_name).set_index('sku_id')
print(f'Loaded {dc_name}: {demand_characteristics.shape}')


## Score against ground truth
ground_truth.csv exists only in this session (produced by generate_data.py). It was never given to Step 5a. This is the ONLY place these two meet.

In [ ]:
from src.portfolio_impact import score_against_ground_truth

truth_path = 'data_primary/_truth/ground_truth.csv'
ground_truth = pd.read_csv(truth_path, dtype={'source_sku_code': str})

scored = score_against_ground_truth(demand_characteristics, ground_truth)

print(f'Matched {len(scored)} of {len(demand_characteristics)} SKUs')
print()
summary = scored.groupby('category').agg(
    encoded_category_bias=('category_chronic_bias', 'first'),
    recovered_l1_median=('chronic_bias_l1', 'median'),
)
print(summary.round(4).to_string())
print()
corr = scored['total_base_chronic_bias'].corr(scored['chronic_bias_l1'])
mae = (scored['chronic_bias_l1'] - scored['total_base_chronic_bias']).abs().mean()
print(f'Per-SKU correlation: {corr:.3f}  |  Mean absolute error: {mae:.4f}')

scored.to_csv('data_primary/clean/step5b_scoring.csv')


## Upload Step 5a's censoring_diagnostics.csv
This is read programmatically -- the flagged-SKU list comes directly from Step 5a's own `verdict` column, no manual transcription. Whatever a given dataset's Step 5a run flags is what this cell picks up automatically.

In [ ]:
from google.colab import files
print('Select censoring_diagnostics.csv:')
uploaded = files.upload()
cd_name = list(uploaded.keys())[0]
censoring_diagnostics = pd.read_csv(cd_name)
print(f'Loaded {cd_name}: {censoring_diagnostics.shape}')
print(censoring_diagnostics['verdict'].value_counts().to_string())


## Portfolio concentration of low-confidence SKUs
flagged_sku_ids is read from censoring_diagnostics.csv above, not typed in.

In [ ]:
from src.portfolio_impact import get_flagged_skus, concentration_report

flagged_sku_ids = get_flagged_skus(censoring_diagnostics)
print(f'{len(flagged_sku_ids)} SKUs flagged INCONCLUSIVE in this run: {sorted(flagged_sku_ids)}')

if not flagged_sku_ids:
    print('No SKUs flagged in this dataset -- nothing further to report.')
else:
    report = concentration_report(demand_characteristics, sku_master, clean_master, flagged_sku_ids)

    print(f"Flagged: {report['n_flagged']} of {report['n_total']} SKUs "
          f"({report['sku_share']:.1%} of count)")
    print(f"  volume share  : {report['volume_share']:.1%}")
    print(f"  revenue share : {report['revenue_share']:.1%}  "
          f"(EUR {report['flagged_annual_revenue_eur']:,.0f} of "
          f"{report['total_annual_revenue_eur']:,.0f})")
    print()
    print('Concentration by axis:')
    for axis, counts in report['axis_concentration'].items():
        print(f'  {axis}:')
        print('   ', counts.to_string().replace(chr(10), chr(10)+'    '))

    report['detail'].to_csv('data_primary/clean/step5b_concentration_detail.csv')


## Download outputs

In [ ]:
from google.colab import files
files.download('data_primary/clean/step5b_scoring.csv')
files.download('data_primary/clean/step5b_concentration_detail.csv')
